In [27]:
import csv

def cargar_datos():
    grafo = {}
    heuristicas = {}

    # Leer Heurísticas
    with open('heuristica.csv', mode='r', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            heuristicas[row['Activity'].strip()] = int(row['Recovery time after burning 300cal (minutes)'])

    # Leer Costos (Grafo)
    with open('funcion_de_costo.csv', mode='r', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            orig = row['Origen'].strip()
            dest = row['Destino'].strip()
            cost = int(row['Cost'])
            if orig not in grafo: grafo[orig] = []
            grafo[orig].append({'dest': dest, 'cost': cost})
            
    return grafo, heuristicas

grafo, heuristicas = cargar_datos()
print("Datos cargados correctamente.")

Datos cargados correctamente.


In [28]:
class TreeNode:
   def __init__(self, state, path_cost=0, heuristic=0, parent=None):
      self.state = state          
      self.path_cost = path_cost  
      self.heuristic = heuristic  
      self.parent = parent        
      self.children = []        

   def add_child(self, child_node):
      #Metodo que conecta a los nodos en el arbol
      self.children.append(child_node)  

   def f(self):
      return self.path_cost + self.heuristic

   def __repr__(self):
      # Representación en texto para poder imprimirlo en consola
      return f"<{self.state}: g={self.path_cost}, h={self.heuristic}, f={self.f()}>"

In [29]:
class PriorityQueue:
   def __init__(self, use_heuristic=False):
      self.items = []
      self.use_heuristic = use_heuristic 

   def EMPTY(self): return len(self.items) == 0
   def POP(self): return self.items.pop(0)
   def ADD(self, item):
      self.items.append(item)
      if self.use_heuristic:
         self.items.sort(key=lambda node: node.f())
      else:
         self.items.sort(key=lambda node: node.path_cost)

In [30]:
def busqueda_gym(inicio, meta, grafo, heuristicas, usar_heuristica=False):
   nodo_raiz = TreeNode(inicio, 0, heuristicas.get(inicio, 0))
   frontera = PriorityQueue(use_heuristic=usar_heuristica)
   frontera.ADD(nodo_raiz)
   
   visitados = set()
   iteracion = 0

   print(f"--- Iniciando Búsqueda ({'A*' if usar_heuristica else 'UCS'}) ---")

   while not frontera.EMPTY():
      nodo_actual = frontera.POP()
      print(f"Iteración {iteracion}: Expandiendo nodo {nodo_actual}")
      iteracion += 1
      
      # Si llegamos a la meta
      if nodo_actual.state == meta:
         print(f"¡Meta encontrada en iteración {iteracion}!")
         return reconstruir_ruta(nodo_actual), nodo_actual.path_cost

      visitados.add(nodo_actual.state)

      # Expandir hijos
      for conexion in grafo.get(nodo_actual.state, []):
         nombre_hijo = conexion['dest']
         costo_arco = conexion['cost']
         
         if nombre_hijo not in visitados:
               hijo = TreeNode(
                  state=nombre_hijo,
                  path_cost=nodo_actual.path_cost + costo_arco,
                  heuristic=heuristicas.get(nombre_hijo, 0),
                  parent=nodo_actual
               )
               nodo_actual.add_child(hijo)
               frontera.ADD(hijo)
               
   return None, 0

def reconstruir_ruta(nodo):
   ruta = []
   actual = nodo
   while actual:
      ruta.append(actual.state)
      actual = actual.parent
   return ruta[::-1] # Invertir para que vaya de inicio a fin

In [31]:
inicio = "Warm-up activities"
meta = "Stretching"

# Ejecutar A*
ruta, costo = busqueda_gym(inicio, meta, grafo, heuristicas, usar_heuristica=True)

print(f"\nRuta recomendada: {' a '.join(ruta)}")
print(f"Costo total de la rutina: {costo} minutos")

--- Iniciando Búsqueda (A*) ---
Iteración 0: Expandiendo nodo <Warm-up activities: g=0, h=5, f=5>
Iteración 1: Expandiendo nodo <Exercise bike: g=10, h=10, f=20>
Iteración 2: Expandiendo nodo <Tread Mill: g=10, h=12, f=22>
Iteración 3: Expandiendo nodo <Step Mill: g=10, h=14, f=24>
Iteración 4: Expandiendo nodo <Skipping Rope: g=10, h=16, f=26>
Iteración 5: Expandiendo nodo <Incline Bench: g=26, h=8, f=34>
Iteración 6: Expandiendo nodo <Dumbbell: g=25, h=9, f=34>
Iteración 7: Expandiendo nodo <Barbell: g=25, h=10, f=35>
Iteración 8: Expandiendo nodo <Incline Bench: g=30, h=8, f=38>
Iteración 9: Expandiendo nodo <Pulling Bars: g=30, h=10, f=40>
Iteración 10: Expandiendo nodo <Climbing Rope: g=36, h=5, f=41>
Iteración 11: Expandiendo nodo <Cable-Crossover: g=35, h=8, f=43>
Iteración 12: Expandiendo nodo <Leg Press Machine: g=35, h=8, f=43>
Iteración 13: Expandiendo nodo <Leg Press Machine: g=37, h=8, f=45>
Iteración 14: Expandiendo nodo <Stretching: g=46, h=0, f=46>
¡Meta encontrada en i